# TaxGPT — Episode 8: The Transformer Block, Assembled

Companion notebook to blog post *"The Transformer Block: Assembling Attention and the Feed-Forward Network (TaxGPT Episode 8)"*.

Combines Episode 5 (multi-head attention), Episode 6 (LayerNorm + residuals), and Episode 7 (feed-forward network) into one stackable `TransformerBlock`.

Reference: Sebastian Raschka, *Build a Large Language Model (From Scratch)*, Ch. 4.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(0)

## 1. The three components, recapped

In [2]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(emb_dim))
        self.beta = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        return self.gamma * (x - mean) / torch.sqrt(var + self.eps) + self.beta


class MultiHeadAttention(nn.Module):
    def __init__(self, emb_dim, n_heads, context_len):
        super().__init__()
        assert emb_dim % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = emb_dim // n_heads
        self.W_q = nn.Linear(emb_dim, emb_dim, bias=False)
        self.W_k = nn.Linear(emb_dim, emb_dim, bias=False)
        self.W_v = nn.Linear(emb_dim, emb_dim, bias=False)
        self.out_proj = nn.Linear(emb_dim, emb_dim)
        self.register_buffer("mask", torch.tril(torch.ones(context_len, context_len)))

    def forward(self, x):
        B, T, C = x.shape
        Q = self.W_q(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        K = self.W_k(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        V = self.W_v(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(self.mask[:T, :T] == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        out = (attn @ V).transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(out)


class FeedForward(nn.Module):
    def __init__(self, emb_dim, hidden_dim):
        super().__init__()
        self.fc1 = nn.Linear(emb_dim, hidden_dim)
        self.gelu = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, emb_dim)

    def forward(self, x):
        return self.fc2(self.gelu(self.fc1(x)))

print("LayerNorm, MultiHeadAttention, FeedForward defined (Episodes 5-7)")

LayerNorm, MultiHeadAttention, FeedForward defined (Episodes 5-7)


## 2. Assembling the TransformerBlock

In [3]:
class TransformerBlock(nn.Module):
    def __init__(self, emb_dim, n_heads, hidden_dim, context_len):
        super().__init__()
        self.norm1 = LayerNorm(emb_dim)
        self.attn = MultiHeadAttention(emb_dim, n_heads, context_len)
        self.norm2 = LayerNorm(emb_dim)
        self.ff = FeedForward(emb_dim, hidden_dim)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))   # sublayer 1: attention
        x = x + self.ff(self.norm2(x))     # sublayer 2: feed-forward
        return x

EMB_DIM, N_HEADS, HIDDEN_DIM, CONTEXT_LEN = 768, 12, 3072, 1024
block = TransformerBlock(EMB_DIM, N_HEADS, HIDDEN_DIM, CONTEXT_LEN)

x = torch.randn(2, 10, EMB_DIM)
out = block(x)

print("input shape: ", x.shape)
print("output shape:", out.shape)
assert x.shape == out.shape, "a stackable block must preserve its input shape"
print("PASS: shape preserved -- this block can be stacked N times")

input shape: 

 torch.Size([2, 10, 768])
output shape: torch.Size([2, 10, 768])
PASS: shape preserved -- this block can be stacked N times


## 3. Confirming both sublayers contribute meaningfully

A quick way to catch a wiring bug: verify each sublayer's contribution is non-trivial for a freshly initialized block.

In [4]:
with torch.no_grad():
    attn_contribution = block.attn(block.norm1(x))
    x_after_attn = x + attn_contribution
    ff_contribution = block.ff(block.norm2(x_after_attn))

print("mean abs attention contribution:   ", attn_contribution.abs().mean().item())
print("mean abs feed-forward contribution:", ff_contribution.abs().mean().item())
print()
print("Both should be comfortably nonzero. A collapse toward 0 on either one signals a wiring bug --")
print("this is exactly the kind of check worth running before a full training run.")

mean abs attention contribution:    0.13936209678649902
mean abs feed-forward contribution: 0.15807034075260162

Both should be comfortably nonzero. A collapse toward 0 on either one signals a wiring bug --
this is exactly the kind of check worth running before a full training run.


## 4. Parameter count for one block

In [5]:
n_params = sum(p.numel() for p in block.parameters())
attn_params = sum(p.numel() for p in block.attn.parameters())
ff_params = sum(p.numel() for p in block.ff.parameters())
norm_params = sum(p.numel() for p in block.norm1.parameters()) + sum(p.numel() for p in block.norm2.parameters())

print(f"total params in one TransformerBlock: {n_params:,}")
print(f"  attention:    {attn_params:,} ({attn_params/n_params:.1%})")
print(f"  feed-forward: {ff_params:,} ({ff_params/n_params:.1%})")
print(f"  layer norms:  {norm_params:,} ({norm_params/n_params:.1%})")

total params in one TransformerBlock: 7,085,568
  attention:    2,360,064 (33.3%)
  feed-forward: 4,722,432 (66.6%)
  layer norms:  3,072 (0.0%)


## 5. Stacking a preview: N blocks, chained

This is next episode's job in full (with embeddings and the output head added), but the shape-preservation property from step 2 is what makes it trivial to preview here.

In [6]:
N_LAYERS = 12  # verify against your actual model -- this is TaxGPT's assumed spec

blocks = nn.Sequential(*[
    TransformerBlock(EMB_DIM, N_HEADS, HIDDEN_DIM, CONTEXT_LEN) for _ in range(N_LAYERS)
])

out = blocks(x)
print(f"output shape after {N_LAYERS} stacked blocks:", out.shape)

total_block_params = sum(p.numel() for p in blocks.parameters())
print(f"total parameters across {N_LAYERS} blocks: {total_block_params:,}")

output shape after 12 stacked blocks: torch.Size([2, 10, 768])
total parameters across 12 blocks: 85,026,816


## Takeaway

The `TransformerBlock` is the repeating unit: multi-head attention and a feed-forward network, each wrapped in pre-norm LayerNorm and a residual connection. Because it preserves input shape exactly, it stacks cleanly — which is literally the entire architecture story for the middle of a GPT-style model.

**Next notebook: Episode 9 — Assembling the full 131M-parameter model (embeddings in front, output head at the end).**